<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pixal3D — Pixel-Aligned 3D Generation (SIGGRAPH 2026)

State-of-the-art image-to-3D by Tencent ARC Lab. Generates high-fidelity GLB meshes with PBR textures from a single image. Built on Microsoft's TRELLIS.2 backbone.

### Quick Start
1. Connect to a GPU runtime (A100, L4, or T4)
2. Run all steps in order — no token, no sign-up needed
3. Launch the Gradio UI (Step 4) or test with a single image (Step 6)

| GPU | VRAM | Resolution | Mode |
|-----|------|-----------|------|
| A100 | 40 GB | 1536 px | Standard |
| L4 | 24 GB | 1024 px | Low-VRAM (recommended) |
| T4 | 15 GB | 1024 px | Low-VRAM (auto) |

Upload images to `/content/images_in/`, then run **Step 7** to batch-process.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs prebuilt CUDA wheels compiled for Colab GPUs from GitHub Releases.

import os, sys, subprocess
import torch

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Auto-detect GPU
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f"{cap[0]}{cap[1]}"
# Map SM arch to release tag suffix
arch_tag = {'75': 't4', '80': 'a100', '89': 'l4'}.get(arch, 't4')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] Detected {gpu_name} ({vram_gb:.0f} GB) \u2192 sm_{arch} ({arch_tag})')
os.makedirs('/content/images_in', exist_ok=True)

# Prebuilt CUDA wheels from GitHub Releases
WHEELS_TAG = f'wheels-{arch_tag}-v1.0'
RELEASE = f'https://github.com/Skquark/AEI-Colab-Notebooks/releases/download/{WHEELS_TAG}'

# Exact wheel filenames (same names across all GPU archs)
WHEELS = {
    'o_voxel': 'o_voxel-0.0.1-cp312-cp312-linux_x86_64.whl',
    'cumesh': 'cumesh-0.0.1-cp312-cp312-linux_x86_64.whl',
    'flex_gemm': 'flex_gemm-1.0.0-cp312-cp312-linux_x86_64.whl',
    'nvdiffrec_render': 'nvdiffrec_render-0.0.0-cp312-cp312-linux_x86_64.whl',
}

for name, whl in WHEELS.items():
    url = f'{RELEASE}/{whl}'
    print(f'  Installing {name}...')
    try:
        subprocess.run(['pip', 'install', '-q', '--no-deps', url], check=True)
    except Exception as e:
        print(f'  [WARN] {name}: {e}')
print('[CUDA Packages] Installed')

# nvdiffrast (compile from NVlabs source \u2014 links against local torch)
print('[nvdiffrast] Compiling from NVlabs source...')
!pip install -q ninja
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
subprocess.run(['pip', 'install', '--no-build-isolation', 'git+https://github.com/NVlabs/nvdiffrast.git'], check=True)
print('[nvdiffrast] Done')

# Python dependencies
!pip install -q trimesh pygltflib plyfile moderngl huggingface_hub kornia kornia-rs \
    imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    zstandard timm diffusers accelerate gradio einops onnxruntime
!pip install -q git+https://github.com/microsoft/MoGe.git

# natten (prebuilt wheel for NAF upsampling)
natten_ok = False
try:
    print(f'[natten] Installing for sm_{arch}...')
    !pip install natten==0.21.6+torch2110cu128 -f https://whl.natten.org -q
    import natten; natten_ok = getattr(natten, 'HAS_LIBNATTEN', False)
    print('  natten - done' if natten_ok else '  natten - no libnatten, NAF disabled')
except Exception:
    print('  natten - skipped (NAF disabled)')

!pip install -q --force-reinstall --no-deps \
    https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl

print(f'\\n[DONE] {gpu_name} ({vram_gb:.0f} GB) \u2014 all dependencies installed')


In [ ]:
# @title STEP 2 — Clone Repo & Pre-cache Models
# @markdown Clones Pixal3D and pre-downloads model weights from HuggingFace (~3-5 GB). Run once — cached for future sessions.
import os, subprocess

os.chdir('/content')
if not os.path.exists('/content/Pixal3D/.git'):
    if os.path.exists('/content/Pixal3D'):
        import shutil; shutil.rmtree('/content/Pixal3D')
    subprocess.run(['git', 'clone', 'https://github.com/TencentARC/Pixal3D.git', '/content/Pixal3D'], check=True)
else:
    print('[Step 2] Pixal3D already cloned')

os.chdir('/content/Pixal3D')
subprocess.run(['git', 'lfs', 'pull'], capture_output=True)

from huggingface_hub import snapshot_download

print('[Cache] DinoV3 feature extractor...')
snapshot_download('camenduru/dinov3-vitl16-pretrain-lvd1689m',
    allow_patterns=['*.bin','*.json','*.txt','*.model','*.safetensors'], max_workers=4)

print('[Cache] BiRefNet (background removal)...')
snapshot_download('ZhengPeng7/BiRefNet',
    allow_patterns=['*.pth','*.json','*.txt','*.bin','*.model'], max_workers=2)

hdri_dir = '/content/Pixal3D/assets/hdri'
if os.path.isdir(hdri_dir):
    exr_files = [f for f in os.listdir(hdri_dir) if f.endswith('.exr')]
    if exr_files: print(f'[Assets] {len(exr_files)} HDRI maps found')
    else: print('[Warning] No .exr files — preview renders use fallback')

print('\n[DONE] All models cached')
os.makedirs('/content/Pixal3D/output', exist_ok=True)

In [ ]:
# @title STEP 3 — Environment & Core Functions
# @markdown Sets environment variables, attention backends, and defines `init_models()` and `run_inference()`.
DECIMATION = 700000 # @param {type:"integer"}
TEXTURE_SIZE = 4096 # @param [2048, 4096] {type:"raw"}

import os, sys, math, time, json, glob, gc
import torch, numpy as np
from PIL import Image
from datetime import datetime

os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Attention backend auto-detect
attn_backend = 'sdpa'
try:
    import flash_attn_interface; attn_backend = 'flash_attn_3'
    print('[Attn] FlashAttention 3')
except ImportError:
    try:
        import flash_attn; attn_backend = 'flash_attn'
        print('[Attn] FlashAttention 2')
    except ImportError:
        print('[Attn] SDPA fallback')
os.environ['ATTN_BACKEND'] = attn_backend
os.environ['SPARSE_ATTN_BACKEND'] = attn_backend
os.environ['FLEX_GEMM_AUTOTUNE_CACHE_PATH'] = '/content/autotune_cache.json'
os.environ['FLEX_GEMM_AUTOTUNER_VERBOSE'] = '1'

import cv2

os.chdir('/content/Pixal3D')
if '/content/Pixal3D' not in sys.path:
    sys.path.insert(0, '/content/Pixal3D')

# Config
MODEL_PATH = 'TencentARC/Pixal3D'
MOGE_MODEL_NAME = 'Ruicheng/moge-2-vitl'

IMAGE_COND_CONFIGS = {
    'ss': {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 512,  'grid_resolution': 16},
    'shape_512':  {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 512,  'grid_resolution': 32,  'use_naf_upsample': True, 'naf_target_size': 512},
    'shape_1024': {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 1024, 'grid_resolution': 64,  'use_naf_upsample': True, 'naf_target_size': 512},
    'tex_1024':   {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 1024, 'grid_resolution': 64,  'use_naf_upsample': True, 'naf_target_size': 1024},
}

if not globals().get('natten_ok', False):
    print('[NAF] NATTEN unavailable — disabling NAF upsampling')
    for k in ('shape_512', 'shape_1024', 'tex_1024'):
        IMAGE_COND_CONFIGS[k] = {**IMAGE_COND_CONFIGS[k], 'use_naf_upsample': False}

pipeline = None
moge_model = None
envmap = None
_low_vram_active = None

def build_image_cond_model(config):
    from pixal3d.trainers.flow_matching.mixins.image_conditioned_proj import DinoV3ProjFeatureExtractor
    m = DinoV3ProjFeatureExtractor(**config); m.eval(); return m

def load_moge_model(device='cuda'):
    from moge.model.v2 import MoGeModel
    m = MoGeModel.from_pretrained(MOGE_MODEL_NAME).to(device); m.eval(); return m

def cleanup_memory():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def init_models(low_vram=False):
    global pipeline, moge_model, envmap, _low_vram_active
    if pipeline is not None and _low_vram_active == low_vram:
        return
    _low_vram_active = low_vram
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{'='*60}")
    print(f'[Init] {gpu_name} ({vram:.0f} GB) | Low-VRAM: {low_vram}')
    print(f"{'='*60}")

    from pixal3d.pipelines import Pixal3DImageTo3DPipeline
    print(f'[Pipeline] Loading {MODEL_PATH}...')
    pipeline = Pixal3DImageTo3DPipeline.from_pretrained(MODEL_PATH)

    print('[ImageCond] Building DinoV3 models...')
    pipeline.image_cond_model_ss = build_image_cond_model(IMAGE_COND_CONFIGS['ss'])
    pipeline.image_cond_model_shape_512 = build_image_cond_model(IMAGE_COND_CONFIGS['shape_512'])
    pipeline.image_cond_model_shape_1024 = build_image_cond_model(IMAGE_COND_CONFIGS['shape_1024'])
    pipeline.image_cond_model_tex_1024 = build_image_cond_model(IMAGE_COND_CONFIGS['tex_1024'])

    if low_vram:
        print('[NAF] Pre-downloading weights (CPU)...')
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()
        pipeline._device = torch.device('cuda')
        pipeline.low_vram = True
    else:
        pipeline.low_vram = False
        pipeline.cuda()
        pipeline.image_cond_model_ss.cuda()
        pipeline.image_cond_model_shape_512.cuda()
        pipeline.image_cond_model_shape_1024.cuda()
        pipeline.image_cond_model_tex_1024.cuda()
        print('[NAF] Pre-loading upsampler...')
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()

    print('[MoGe-2] Loading camera model...')
    moge_model = load_moge_model(device='cpu' if low_vram else 'cuda')

    print('[EnvMap] Loading HDRI...')
    from pixal3d.renderers import EnvMap
    dev = 'cpu' if low_vram else 'cuda'
    envmap = {}
    _base = '/content/Pixal3D/assets/hdri'
    for name in ['forest','sunset','courtyard']:
        p = os.path.join(_base, f'{name}.exr')
        if os.path.exists(p):
            envmap[name] = EnvMap(torch.tensor(cv2.cvtColor(cv2.imread(p, cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB), dtype=torch.float32, device=dev))
    print(f'[Init] {len(envmap)} HDRI maps loaded. Ready!')

def compute_f_pixels(camera_angle_x, resolution):
    fl = 16.0 / torch.tan(torch.tensor(camera_angle_x / 2.0))
    return float((fl * resolution / 32.0).item())

def distance_from_fov(camera_angle_x, grid_point, target_point, mesh_scale, image_resolution):
    R = torch.tensor([[1.,0.,0.],[0.,0.,-1.],[0.,1.,0.]])
    gp = grid_point.float() @ R.T / mesh_scale / 2
    xt, yt = float(target_point[0].item()), float(target_point[1].item())
    fp = compute_f_pixels(camera_angle_x, image_resolution)
    return {'distance_from_x': float(fp * gp[0].item() / (xt - image_resolution/2) - gp[1].item()), 'f_pixels': fp}

def get_camera_params(image_path, moge, device='cuda', mesh_scale=1.0, extend_pixel=0, image_resolution=512):
    global _low_vram_active
    if _low_vram_active: moge.to(device)
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    t = torch.from_numpy(np.array(img).astype(np.float32)/255.).permute(2,0,1).to(device)
    with torch.no_grad(): out = moge.infer(t)
    if _low_vram_active: moge.cpu(); cleanup_memory()
    fx_norm = out['intrinsics'].squeeze().cpu().numpy()[0,0]
    camera_angle_x = 2 * math.atan(w / (2 * fx_norm * w))
    d = distance_from_fov(camera_angle_x, torch.tensor([-1.,0.,0.]),
        torch.tensor([0-extend_pixel, image_resolution-1+extend_pixel]), mesh_scale, image_resolution)
    return {'camera_angle_x': camera_angle_x, 'distance': d['distance_from_x'], 'mesh_scale': mesh_scale}

def run_inference(image_path, seed=42, resolution=1536, low_vram=False,
                  ss_steps=12, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
                  shape_steps=12, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
                  tex_steps=12, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
        manual_fov=-1.0, fov_unit='deg', render_mode='shaded_forest',
                  output_path=None, render_preview=False, verbose=True):
    global pipeline, moge_model, envmap
    t0 = time.time()
    if pipeline is None: init_models(low_vram=low_vram)
    if verbose: print(f'[Inference] {os.path.basename(image_path)}')
    img = Image.open(image_path).convert('RGBA')
    image_preprocessed = pipeline.preprocess_image(img)
    if manual_fov > 0:
        if fov_unit == 'rad':
            camera_angle_x = float(manual_fov)
        else:
            camera_angle_x = math.radians(manual_fov)
        d = distance_from_fov(camera_angle_x, torch.tensor([-1.,0.,0.]),
            torch.tensor([0, 511]), 1.0, 512)
        cam = {'camera_angle_x': camera_angle_x, 'distance': d['distance_from_x'], 'mesh_scale': 1.0}
        if verbose: print(f"  Camera (manual): fov={math.degrees(camera_angle_x):.1f}°, dist={cam['distance']:.3f}")
    else:
        tmp = '/content/_tmp_cam.png'; image_preprocessed.save(tmp)
        cam = get_camera_params(tmp, moge_model, mesh_scale=1.0, extend_pixel=0, image_resolution=512)
        os.remove(tmp)
        if verbose: print(f"  Camera (auto): fov={math.degrees(cam['camera_angle_x']):.1f}°, dist={cam['distance']:.3f}")
    torch.manual_seed(int(seed))
    hr_res = int(resolution)
    ss = {'steps': int(ss_steps), 'guidance_strength': float(ss_guidance),
          'guidance_rescale': float(ss_rescale), 'rescale_t': float(ss_rescale_t)}
    sh = {'steps': int(shape_steps), 'guidance_strength': float(shape_guidance),
          'guidance_rescale': float(shape_rescale), 'rescale_t': float(shape_rescale_t)}
    tx = {'steps': int(tex_steps), 'guidance_strength': float(tex_guidance),
          'guidance_rescale': float(tex_rescale), 'rescale_t': float(tex_rescale_t)}
    pipe_type = f'{hr_res}_cascade'
    if verbose: print(f'  Pipeline: {pipe_type}')
    mesh_list, (shape_slat, tex_slat, res) = pipeline.run(
        image_preprocessed, camera_params=cam, seed=int(seed),
        sparse_structure_sampler_params=ss, shape_slat_sampler_params=sh,
        tex_slat_sampler_params=tx, preprocess_image=False, return_latent=True,
        pipeline_type=pipe_type, max_num_tokens=49152)
    import o_voxel
    mesh = mesh_list[0]
    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,
        coords=mesh.coords, attr_layout=pipeline.pbr_attr_layout,
        grid_size=res, aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=DECIMATION, texture_size=TEXTURE_SIZE,
        remesh=True, remesh_band=1, remesh_project=0, use_tqdm=True)
    rot = np.array([[-1,0,0,0],[0,0,-1,0],[0,-1,0,0],[0,0,0,1]], dtype=np.float64)
    glb.apply_transform(rot)
    if output_path is None:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        name = os.path.splitext(os.path.basename(image_path))[0]
        output_path = f'/content/Pixal3D/output/{name}_{ts}.glb'
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    glb.export(output_path, extension_webp=True)
    preview_path = None
    if render_preview:
        try:
            from pixal3d.utils import render_utils
            from pixal3d.renderers import EnvMap
            cd = cam['distance']
            if envmap:
                if low_vram:
                    for v in envmap.values():
                        v.image = v.image.cuda()
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                mesh.simplify(16777216)
                renders = render_utils.render_proj_aligned_video(
                    mesh, camera_angle_x=cam['camera_angle_x'], distance=cd,
                    resolution=512, num_frames=8, envmap=envmap,
                    near=max(0.01, cd-2.0), far=cd+10.0)
                if low_vram:
                    for v in envmap.values():
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                        v.image = v.image.cpu()
                    torch.cuda.empty_cache()
                k = render_mode if render_mode in renders else list(renders.keys())[0]
                preview_path = output_path.replace('.glb','_preview.png')
                Image.fromarray(renders[k][len(renders[k])//2]).save(preview_path)
        except Exception as e:
            if verbose: print(f'  [Preview skip] {e}')
    # Clear VRAM used by mesh/renders. Keeps pipeline loaded for next call.
    del mesh, mesh_list, shape_slat, tex_slat
    cleanup_memory()
    if verbose: print(f'  [Done] {output_path} ({time.time()-t0:.0f}s)')
    return output_path, preview_path

print('[Step 3] Core functions ready.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web interface for single-image 3D generation. Click the `.gradio.live` link to open in a new tab.
import sys, os
if '/content/Pixal3D' not in sys.path:
    sys.path.insert(0, '/content/Pixal3D')
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

import gradio as gr
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_name = torch.cuda.get_device_name(0)
auto_lv = vram_total < 23
auto_res = 1024 if auto_lv else 1536

def generate_wrapper(image_input, seed, resolution, low_vram, manual_fov, fov_unit, render_mode,
                     ss_steps, ss_guidance, ss_rescale, ss_rescale_t,
                     shape_steps, shape_guidance, shape_rescale, shape_rescale_t,
                     tex_steps, tex_guidance, tex_rescale, tex_rescale_t,
                     progress=gr.Progress()):
    if '/content/Pixal3D' not in sys.path:
        sys.path.insert(0, '/content/Pixal3D')
    global pipeline
    progress(0, desc='Initializing...')
    if pipeline is None: init_models(low_vram=low_vram)
    progress(0.3, desc='Generating 3D...')
    glb_path, prev_path = run_inference(
        image_input, seed=seed, resolution=resolution, low_vram=low_vram,
        ss_steps=ss_steps, ss_guidance=ss_guidance, ss_rescale=ss_rescale, ss_rescale_t=ss_rescale_t,
        shape_steps=shape_steps, shape_guidance=shape_guidance, shape_rescale=shape_rescale, shape_rescale_t=shape_rescale_t,
        tex_steps=tex_steps, tex_guidance=tex_guidance, tex_rescale=tex_rescale, tex_rescale_t=tex_rescale_t,
        output_path=None, manual_fov=manual_fov, fov_unit=fov_unit, render_mode=render_mode, render_preview=True, verbose=True)
    progress(0.9, desc='Done!')
    return glb_path, prev_path

demo = gr.Blocks(title='Pixal3D — MissingLink', theme=gr.themes.Soft())
with demo:
    gr.Markdown('# Pixal3D — Pixel-Aligned 3D Generation')
    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type='filepath', label='Input Image', height=280)
            with gr.Accordion('Settings', open=True):
                seed = gr.Number(label='Seed', value=42, precision=0)
                resolution = gr.Radio(['1024','1536'], label='Resolution', value=str(auto_res))
                low_vram = gr.Checkbox(label='Low VRAM Mode', value=auto_lv)
                render_mode = gr.Dropdown(['shaded_forest','shaded_sunset','shaded_courtyard'], value='shaded_forest', label='Preview Render')
            with gr.Accordion('Camera', open=False):
                manual_fov = gr.Slider(-1, 180, value=-1, step=1, label='Manual FOV (°) — -1 = auto')
                fov_unit = gr.Radio(['deg'], label='FOV Unit', value='deg')
            with gr.Accordion('Advanced Sampling', open=False):
                gr.Markdown('**Sparse Structure**')
                ss_steps = gr.Slider(4, 20, value=12, step=1, label='Steps')
                ss_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label='CFG')
                ss_resc = gr.Slider(0.0, 1.0, value=0.7, step=0.1, label='Rescale')
                ss_rt = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='Rescale T')
                gr.Markdown('**Shape**')
                sh_steps = gr.Slider(4, 20, value=8 if auto_lv else 12, step=1, label='Steps')
                sh_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label='CFG')
                sh_resc = gr.Slider(0.0, 1.0, value=0.5, step=0.1, label='Rescale')
                sh_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label='Rescale T')
                gr.Markdown('**Texture**')
                tx_steps = gr.Slider(4, 20, value=8 if auto_lv else 12, step=1, label='Steps')
                tx_cfg = gr.Slider(1.0, 15.0, value=1.0, step=0.5, label='CFG')
                tx_resc = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label='Rescale')
                tx_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label='Rescale T')
            btn = gr.Button('Generate 3D Mesh', variant='primary', size='lg')
        with gr.Column(scale=1):
            preview = gr.Image(label='Turntable Preview', height=280)
            out_file = gr.File(label='Download GLB', file_types=['.glb'])
            gr.Markdown(f'**GPU:** {gpu_name} ({vram_total:.0f} GB) | **Mode:** {"Low-VRAM" if auto_lv else "Standard"}')
    btn.click(fn=generate_wrapper,
        inputs=[img_in,seed,resolution,low_vram,manual_fov,fov_unit,render_mode,ss_steps,ss_cfg,ss_resc,ss_rt,sh_steps,sh_cfg,sh_resc,sh_rt,tx_steps,tx_cfg,tx_resc,tx_rt],
        outputs=[out_file,preview])
    gr.Markdown('**Tips:** Clean-background images work best. Lower steps = faster. GLBs include PBR textures.')

demo.queue(max_size=3).launch(share=True, debug=True, show_error=True)

In [ ]:
# @title Step 5 — Keep Alive
# @markdown Keeps the Colab session active during long inference runs. Run this anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(300)
        print('\u200b', end='', flush=True)
try:
    t = threading.Thread(target=_keep, daemon=True); t.start()
    print('[KeepAlive] Active (pings every 5 min)')
except Exception as e:
    print(f'[KeepAlive] Error: {e}')


In [ ]:
# @title Step 6 — Single Image Inference
# @markdown Generate one GLB from a single image without the UI. Good for quick testing.

IMAGE_PATH = '/content/Pixal3D/assets/images/1_img.png' #@param {type:"string"}
SEED = 42 #@param {type:"integer"}
RESOLUTION = 1536 #@param [1024, 1536] {type:"raw"}
STEPS = 12 # @param {type:"integer"}
LOW_VRAM = True # @param {type:"boolean"}
MANUAL_FOV = -1 # @param {type:"number"}
FOV_UNIT = 'deg' # @param ['deg', 'rad'] {type:'raw'}
RENDER_MODE = 'shaded_forest' # @param ['shaded_forest', 'shaded_sunset', 'shaded_courtyard'] {type:'raw'}

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
quality_mode = vram >= 30   # A100: full quality. L4/T4: low-VRAM
steps = STEPS

if os.path.exists(IMAGE_PATH):
    glb_path, preview_path = run_inference(
        IMAGE_PATH, seed=SEED, resolution=RESOLUTION, low_vram=LOW_VRAM,
        ss_steps=steps, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
        shape_steps=steps, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
        tex_steps=steps, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
        manual_fov=MANUAL_FOV, fov_unit=FOV_UNIT, render_mode=RENDER_MODE,
        render_preview=True, verbose=True)
    if preview_path:
        from IPython.display import display, Image as IPImage
        display(IPImage(filename=preview_path))
else:
    print(f'Image not found: {IMAGE_PATH}')
    print('Upload images to /content/images_in/ or use a built-in test image.')


In [ ]:
# @title Step 7 — Batch Process Images
# @markdown Upload images to `/content/images_in/`, then run this cell. Outputs go to Google Drive under `pixal3d_batch_output/`.

INPUT_DIR = '/content/images_in' #@param {type:"string"}
OUTPUT_DIR = '/content/drive/MyDrive/pixal3d_batch_output' #@param {type:"string"}
RESOLUTION = 1536 #@param [1024, 1536] {type:"raw"}
STEPS = 12 # @param {type:"integer"}
LOW_VRAM = True # @param {type:"boolean"}
MANUAL_FOV = -1 # @param {type:"number"}
FOV_UNIT = 'deg' # @param ['deg', 'rad'] {type:'raw'}
RENDER_MODE = 'shaded_forest' # @param ['shaded_forest', 'shaded_sunset', 'shaded_courtyard'] {type:'raw'}

import shutil, glob, os
from datetime import datetime
import torch
os.makedirs(OUTPUT_DIR, exist_ok=True)

images = []
for ext in ['*.png','*.jpg','*.jpeg','*.webp','*.PNG','*.JPG','*.JPEG']:
    images.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
images = sorted(set(images))

if not images:
    print(f'[Batch] No images found in {INPUT_DIR}/')
    print('  Upload images using the Colab file sidebar -> /content/images_in/')
else:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    quality_mode = vram_gb >= 30
    #lv = not quality_mode
    #steps = 12 if quality_mode else 8
    lv = LOW_VRAM
    steps = STEPS
    print(f'[Batch] {len(images)} images | {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)')
    print(f'[Batch] {RESOLUTION}px | {"Quality mode" if quality_mode else "Low-VRAM"} | {steps} steps')
    print(f'[Batch] Output: {OUTPUT_DIR}/\n')

    results = []
    for i, img_path in enumerate(images):
        name = os.path.splitext(os.path.basename(img_path))[0]
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        out_path = os.path.join(OUTPUT_DIR, f'{name}_{ts}.glb')
        existing = glob.glob(os.path.join(OUTPUT_DIR, f'{name}_*.glb'))
        if existing:
            print(f'  [{i+1}/{len(images)}] {name} - SKIP')
            results.append((name, 'skipped', existing[0]))
            continue
        print(f'  [{i+1}/{len(images)}] {name} - generating...')
        try:
            glb_path, prev_path = run_inference(
                img_path, seed=42 + i, resolution=RESOLUTION, low_vram=lv,
                ss_steps=steps, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
                shape_steps=steps, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
                tex_steps=steps, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
                manual_fov=MANUAL_FOV, fov_unit=FOV_UNIT, render_mode=RENDER_MODE, render_preview=True, verbose=False)
            shutil.move(glb_path, out_path)
            if prev_path:
                shutil.move(prev_path, out_path.replace('.glb','_preview.png'))
            results.append((name, 'ok', out_path))
        except Exception as e:
            print(f'    [ERROR] {e}')
            results.append((name, 'error', str(e)))
        # Clean VRAM between batch items (keeps pipeline loaded)
        cleanup_memory()

    ok = sum(1 for r in results if r[1] == 'ok')
    sk = sum(1 for r in results if r[1] == 'skipped')
    er = sum(1 for r in results if r[1] == 'error')
    print(f'\n[Batch] Done! {ok} generated, {sk} skipped, {er} errors')
    print(f'[Batch] Output: {OUTPUT_DIR}/')
